In [37]:
from statistics import stdev
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback

import mlflow
from PIL.features import features

from rl_trading_lab.environment.trading_env import TradingEnv, Action

In [38]:
import logging

# Configure logging
logging.basicConfig(
  level=logging.DEBUG,
  format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

- Create a small pandas DataFrame suitable for initializing TradingEnv.
- Columns: timestamp, open, high, low, close, volume.
- Generate 200 rows of plausible OHLCV data with a datetime index.

In [39]:
import pandas as pd
import numpy as np

np.random.seed(42)

features = [
    "ratio_sma_5_close", "ratio_sma_20_close", "ratio_range_close", "fracdiff_0.4_zscore"
]
columns = ["timestamp", "close"] + features

df = pd.read_parquet("../sample_data/btcusdt_sample_10k.parquet",
                     columns=columns
                     )
df

,timestamp,close,ratio_sma_5_close,ratio_sma_20_close,ratio_range_close,fracdiff_0.4_zscore
0,2025-10-21 11:06:35.585,108308.24,1.005122,1.030419,0.000738,0.626690
1,2025-10-21 11:07:46.515,108290.04,1.008869,1.033301,0.000335,0.883138
2,2025-10-21 11:09:18.691,108227.71,1.006301,1.025187,0.000576,0.609088
3,2025-10-21 11:10:02.700,108179.22,1.028342,1.054175,0.000535,-0.021087
4,2025-10-21 11:11:03.722,108256.24,1.059898,1.055163,0.000711,-0.194621
...,...,...,...,...,...,...
9995,2025-10-25 09:38:07.057,111698.93,0.978579,0.993264,0.000293,1.264441
9996,2025-10-25 09:38:50.760,111722.63,0.996468,0.996946,0.000493,1.526022
9997,2025-10-25 09:41:23.762,111699.89,1.020242,1.012118,0.000353,-0.750411
9998,2025-10-25 09:44:27.781,111698.70,1.029383,1.014314,0.000411,-0.594010


In [40]:
df.sort_values(by="timestamp", inplace=True)
df

,timestamp,close,ratio_sma_5_close,ratio_sma_20_close,ratio_range_close,fracdiff_0.4_zscore
0,2025-10-21 11:06:35.585,108308.24,1.005122,1.030419,0.000738,0.626690
1,2025-10-21 11:07:46.515,108290.04,1.008869,1.033301,0.000335,0.883138
2,2025-10-21 11:09:18.691,108227.71,1.006301,1.025187,0.000576,0.609088
3,2025-10-21 11:10:02.700,108179.22,1.028342,1.054175,0.000535,-0.021087
4,2025-10-21 11:11:03.722,108256.24,1.059898,1.055163,0.000711,-0.194621
...,...,...,...,...,...,...
9995,2025-10-25 09:38:07.057,111698.93,0.978579,0.993264,0.000293,1.264441
9996,2025-10-25 09:38:50.760,111722.63,0.996468,0.996946,0.000493,1.526022
9997,2025-10-25 09:41:23.762,111699.89,1.020242,1.012118,0.000353,-0.750411
9998,2025-10-25 09:44:27.781,111698.70,1.029383,1.014314,0.000411,-0.594010


In [41]:
df.dropna(axis=0, inplace=True)

In [42]:
def make_env():
    return TradingEnv(df,
                      lookback_window=0,
                      randomize_start=True,
                      one_trade_mode=True,
                      hold_closes_position=True,
                      reward_type="returns",
                      min_holding_period=2,
                      min_episode_length=2,
                      max_position_pct=0.2,
                      price_column="close",
                      discrete_actions=True,
                      commission_rate=0.0,
                      slippage_rate=0.0,
                      features_to_use=features,
                      )

In [43]:
env = make_env()

2025-10-30 08:20:05,369 - rl_trading_lab.environment.trading_env - INFO - TradingEnv initialized: randomize_start=True, hold_closes_position=True, one_trade_mode=True, min_episode_length=2, reward_type=returns, data_length=9954


In [44]:
obs = env.reset()
obs

2025-10-30 08:20:05,980 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 251/9953


(array([1.0447077e+00, 1.0371249e+00, 4.0128411e-04, 6.7711836e-01,
        0.0000000e+00, 0.0000000e+00, 1.0000000e+00], dtype=float32),
 {'step': np.int64(251),
  'cash': 10000,
  'portfolio_value': 10000,
  'position': 0.0,
  'total_return': 0.0,
  'num_trades': 0})

In [45]:
obs, reward, terminated, truncated, info = env.step(action=Action.BUY)

2025-10-30 08:20:07,969 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=1.0, price=108750.88
2025-10-30 08:20:07,969 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: LONG 0.0184 @ $108750.88 (cash flow: -$2000.00, remaining: $8000.00)
2025-10-30 08:20:07,970 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=0.0184


In [46]:
obs

array([ 1.0437194e+00,  1.0423387e+00,  8.8721601e-04,  3.4255561e-01,
        1.8390656e-02, -8.8633766e-04,  9.9982274e-01], dtype=float32)

In [47]:
reward

np.float64(-0.00017726753107654076)

In [48]:
terminated

False

In [49]:
info

{'step': np.int64(252),
 'cash': np.float64(8000.0),
 'portfolio_value': np.float64(9998.227324689235),
 'position': np.float64(0.018390655781360114),
 'total_return': np.float64(-0.00017726753107654076),
 'num_trades': 1}

In [50]:
info['position'] * obs[3]

np.float64(0.006299822352254406)

In [51]:
env.get_trade_history()

[]

In [52]:
(9972.511419154229 / 10_000 - 1)

-0.0027488580845771438

In [53]:
obs, reward, terminated, truncated, info = env.step(action=Action.SELL)

2025-10-30 08:20:19,544 - rl_trading_lab.environment.portfolio - DEBUG - Position held for 1 bars, need 2 bars minimum


In [54]:
env.get_trade_history()

[]

In [55]:
obs

array([ 1.0644850e+00,  1.0342150e+00,  9.5432578e-04, -1.0701023e+00,
        1.8390656e-02, -1.0961751e-03,  9.9978077e-01], dtype=float32)

In [56]:
info

{'step': np.int64(253),
 'cash': np.float64(8000.0),
 'portfolio_value': np.float64(9997.807649924303),
 'position': np.float64(0.018390655781360114),
 'total_return': np.float64(-0.0002192350075696595),
 'num_trades': 1,
 'sharpe': np.float64(-25.720927809752755),
 'max_drawdown': np.float64(4.197491728307265e-05)}

In [57]:
from stable_baselines3.common.vec_env import VecFrameStack
vec_env = VecFrameStack(env, n_stack=4)

AttributeError: 'TradingEnv' object has no attribute 'num_envs'

In [58]:

m_vec = Monitor(env, filename='monitor.csv')

In [59]:
m_vec.reset()

2025-10-30 08:20:26,247 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 3730/9953


(array([ 1.0597899e+00,  1.0524679e+00,  1.9065179e-04, -3.7684655e-01,
         0.0000000e+00,  0.0000000e+00,  1.0000000e+00], dtype=float32),
 {'step': np.int64(3730),
  'cash': 10000,
  'portfolio_value': 10000,
  'position': 0.0,
  'total_return': 0.0,
  'num_trades': 0})

In [60]:
def run_environment_random(make_env, n_steps=1000):
    _env = Monitor(make_env(), filename='monitor.csv')
    _env.reset()
    for _ in range(n_steps):
        action = _env.action_space.sample()
        obs, reward, terminated, truncated, info = _env.step(action)
        if terminated or truncated:
            _env.reset()

    return _env.get_episode_times(), _env.get_episode_rewards(), _env.get_episode_lengths()

In [ ]:
t, r, l = run_environment_random(make_env=make_env, n_steps=1000)

In [61]:
from statistics import mean, stdev
mean(r), stdev(r)

NameError: name 'r' is not defined

In [62]:
mean(l), stdev(l)

NameError: name 'l' is not defined

In [ ]:
# episodes = []
# for _ in range(100):
#     actions = []
#     rewards = []
#     action = m_vec.action_space.sample()
#     obs, reward, terminated, truncated, info = m_vec.step(action)
#
#     actions.append(action)
#     rewards.append(reward)
#
#     if terminated or truncated:
#         m_vec.reset()
#         episodes.append({'actions': actions, 'rewards': rewards})

In [ ]:

model = A2C("MlpPolicy", make_env(), verbose=1, tensorboard_log="tb_log")

In [ ]:
model.learn(total_timesteps=100_000)

In [63]:
def run_environment_model(make_env, model, n_steps=1000):
    _env = Monitor(make_env(), filename='monitor.csv')
    obs, info = _env.reset()
    for _ in range(n_steps):
        action, _ = model.predict(obs)
        obs, reward, terminated, truncated, info = _env.step(action)
        if terminated or truncated:
            obs, info = _env.reset()

    return _env.get_episode_times(), _env.get_episode_rewards(), _env.get_episode_lengths()

In [ ]:
t, r, l = run_environment_model(make_env=make_env, model=model, n_steps=1000)

In [ ]:
t, r, l

In [ ]:
mean(r), stdev(r)

In [ ]:
mean(l), stdev(l)

In [ ]:
test_env = Monitor(make_env(), filename='monitor_test.csv')
obs, info = test_env.reset()
episodes = []
actions = []
rewards = []
for _ in range(100):

    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, info = test_env.step(action)

    actions.append(action)
    rewards.append(reward)

    if terminated or truncated:
        test_env.reset()
        episodes.append({'actions': actions.copy(), 'rewards': rewards.copy()})
        actions = []
        rewards = []

In [ ]:
episodes

In [ ]:
m_vec.get_total_steps()

In [ ]:
test_env.get_total_steps()

In [ ]:
test_env.get_episode_rewards()

In [ ]:
test_env.get_episode_lengths()

## Use Monitor Wrapper

In [ ]:
env = Monitor(make_env())
action = env.action_space.sample()

In [ ]:
action

In [ ]:
obs, info = env.reset()
env.step(action)

In [ ]:
model.predict(obs)

## Use VecNormalize

In [65]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

In [64]:
def make_monitored_env():
    return Monitor(make_env(), filename='monitor.csv')

In [16]:
env = VecNormalize(DummyVecEnv([make_monitored_env]))

2025-10-30 07:44:43,355 - rl_trading_lab.environment.trading_env - INFO - TradingEnv initialized: randomize_start=True, hold_closes_position=True, one_trade_mode=True, min_episode_length=2, reward_type=returns, data_length=9954


In [ ]:
env

In [ ]:
obs = env.reset()
for _ in range(100):
    action = env.action_space.sample()
    # obs, reward, terminated, truncated, info = env.step(action)
    obs, reward, done, info = env.step([action])
    if terminated or truncated:
        env.reset()


In [ ]:
env.get_original_reward()

In [ ]:
env.env_method('get_episode_rewards')

In [ ]:
env.env_method('get_episode_lengths')

In [66]:
import os; os.getcwd()

'/Users/mohamedali/trading_project/rl-trading-lab/notebooks'

In [67]:

mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("mlflow_test")  # or reuse "rl_trading"
mlflow.autolog()

2025/10/30 08:20:59 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2025/10/30 08:20:59 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.


In [19]:
model = A2C(
    "MlpPolicy",
    env=VecNormalize(DummyVecEnv([make_monitored_env]), ),
    verbose=1,
    tensorboard_log="./tb_log/"
)
model.learn(total_timesteps=100_000)

2025-10-30 07:45:16,771 - rl_trading_lab.environment.trading_env - INFO - TradingEnv initialized: randomize_start=True, hold_closes_position=True, one_trade_mode=True, min_episode_length=2, reward_type=returns, data_length=9954


Using cpu device


2025-10-30 07:45:17,324 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 3451/9953


Logging to ./tb_log/A2C_4


2025-10-30 07:45:17,328 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=-1.0, price=108006.56
2025-10-30 07:45:17,329 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: SHORT 0.0185 @ $108006.56 (cash flow: +$2000.00, -$0.00, remaining: $12000.00)
2025-10-30 07:45:17,329 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=-0.0185
2025-10-30 07:45:17,332 - rl_trading_lab.environment.portfolio - DEBUG - Position held for 1 bars, need 2 bars minimum
2025-10-30 07:45:17,334 - rl_trading_lab.environment.portfolio - DEBUG - Closing position: current=-0.0185, signal=1.0
2025-10-30 07:45:17,334 - rl_trading_lab.environment.portfolio - DEBUG - Position closed: P&L=$1.97, Commission=$0.00, Net=$1.97 (cash flow: -$1998.03, balance: $10001.97)
2025-10-30 07:45:17,335 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 7136/9953
2025-10-30 07:45:17,353 - rl_trading_lab.environment.portfolio - DEBUG - Opening

...

-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 584       |
|    ep_rew_mean        | 0.000271  |
| time/                 |           |
|    fps                | 912       |
|    iterations         | 19000     |
|    time_elapsed       | 104       |
|    total_timesteps    | 95000     |
| train/                |           |
|    entropy_loss       | -1.68e-05 |
|    explained_variance | -0.289    |
|    learning_rate      | 0.0007    |
|    n_updates          | 18999     |
|    policy_loss        | 3.05e-08  |
|    value_loss         | 0.0219    |
-------------------------------------


2025-10-30 07:47:01,557 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 3773/9953
2025-10-30 07:47:01,560 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=1.0, price=108088.04
2025-10-30 07:47:01,561 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: LONG 0.0185 @ $108088.04 (cash flow: -$2000.00, remaining: $8000.00)
2025-10-30 07:47:01,561 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=0.0185
2025-10-30 07:47:01,563 - rl_trading_lab.environment.portfolio - DEBUG - Position held for 1 bars, need 2 bars minimum
2025-10-30 07:47:01,565 - rl_trading_lab.environment.trading_env - DEBUG - Hold action closing position: size=0.0185
2025-10-30 07:47:01,565 - rl_trading_lab.environment.portfolio - DEBUG - Position closed: P&L=$1.61, Commission=$0.00, Net=$1.61 (cash flow: +$2001.61, -$0.00, balance: $10001.61)
2025-10-30 07:47:01,566 - rl_trading_lab.environment.trading_env - DEBUG - Episode res

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 563      |
|    ep_rew_mean        | 0.000188 |
| time/                 |          |
|    fps                | 913      |
|    iterations         | 19100    |
|    time_elapsed       | 104      |
|    total_timesteps    | 95500    |
| train/                |          |
|    entropy_loss       | -4.9e-05 |
|    explained_variance | 0.134    |
|    learning_rate      | 0.0007   |
|    n_updates          | 19099    |
|    policy_loss        | 8.25e-07 |
|    value_loss         | 0.0669   |
------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 563       |
|    ep_rew_mean        | 0.000188  |
| time/                 |           |
|    fps                | 914       |
|    iterations         | 19200     |
|    time_elapsed       | 104       |
|    total_timesteps    | 96000     |
| train/                |    

2025-10-30 07:47:04,305 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 2082/9953
2025-10-30 07:47:04,309 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=1.0, price=112038.15
2025-10-30 07:47:04,309 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: LONG 0.0179 @ $112038.15 (cash flow: -$2000.00, remaining: $8000.00)
2025-10-30 07:47:04,309 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=0.0179
2025-10-30 07:47:04,311 - rl_trading_lab.environment.portfolio - DEBUG - Position held for 1 bars, need 2 bars minimum
2025-10-30 07:47:04,314 - rl_trading_lab.environment.trading_env - DEBUG - Hold action closing position: size=0.0179
2025-10-30 07:47:04,314 - rl_trading_lab.environment.portfolio - DEBUG - Position closed: P&L=$-0.35, Commission=$0.00, Net=$-0.35 (cash flow: +$1999.65, -$0.00, balance: $9999.65)
2025-10-30 07:47:04,315 - rl_trading_lab.environment.trading_env - DEBUG - Episode re

-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 584       |
|    ep_rew_mean        | 0.000104  |
| time/                 |           |
|    fps                | 918       |
|    iterations         | 19700     |
|    time_elapsed       | 107       |
|    total_timesteps    | 98500     |
| train/                |           |
|    entropy_loss       | -5.15e-05 |
|    explained_variance | -0.202    |
|    learning_rate      | 0.0007    |
|    n_updates          | 19699     |
|    policy_loss        | -1.9e-06  |
|    value_loss         | 0.0923    |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 584       |
|    ep_rew_mean        | 0.000104  |
| time/                 |           |
|    fps                | 919       |
|    iterations         | 19800     |
|    time_elapsed       | 107       |
|    total_timesteps    | 99000     |
| train/    

In [68]:
from typing import Any, Union
import sys
import numpy as np
from stable_baselines3.common.logger import HumanOutputFormat, KVWriter, Logger

class MLflowOutputFormat(KVWriter):
    """
    Dumps key/value pairs into MLflow's numeric format.
    """

    def write(
        self,
        key_values: dict[str, Any],
        key_excluded: dict[str, Union[str, tuple[str, ...]]],
        step: int = 0,
    ) -> None:

        for (key, value), (_, excluded) in zip(
            sorted(key_values.items()), sorted(key_excluded.items())
        ):

            if excluded is not None and "mlflow" in excluded:
                continue

            if isinstance(value, np.ScalarType):
                if not isinstance(value, str):
                    mlflow.log_metric(key, value, step)


loggers = Logger(
    folder=None,
    output_formats=[HumanOutputFormat(sys.stdout), MLflowOutputFormat()],
)

In [71]:
# Create log dir where evaluation results will be saved
eval_log_dir = "./eval_logs/"
os.makedirs(eval_log_dir, exist_ok=True)
eval_env = make_monitored_env()
n_training_envs = 1

2025-10-30 08:21:29,722 - rl_trading_lab.environment.trading_env - INFO - TradingEnv initialized: randomize_start=True, hold_closes_position=True, one_trade_mode=True, min_episode_length=2, reward_type=returns, data_length=9954


In [72]:
# Create callback that evaluates agent for 5 episodes every 500 training environment steps.
# When using multiple training environments, agent will be evaluated every
# eval_freq calls to train_env.step(), thus it will be evaluated every
# (eval_freq * n_envs) training steps. See EvalCallback doc for more information.
eval_callback = EvalCallback(eval_env, best_model_save_path=eval_log_dir,
                              log_path=eval_log_dir, eval_freq=max(1000 // n_training_envs, 1),
                              n_eval_episodes=5, deterministic=True,
                              render=False)

In [73]:
with mlflow.start_run(run_name="A2C"):
    model = A2C(
        "MlpPolicy",
        env=make_monitored_env(),
        verbose=1,
        tensorboard_log="./tb_log/"
    )
    model.set_logger(loggers)
    model.learn(total_timesteps=10000, log_interval=100, callback=eval_callback)
    # mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=10)
    print()

2025-10-30 08:22:21,293 - rl_trading_lab.environment.trading_env - INFO - TradingEnv initialized: randomize_start=True, hold_closes_position=True, one_trade_mode=True, min_episode_length=2, reward_type=returns, data_length=9954
2025-10-30 08:22:21,297 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 4819/9953
2025-10-30 08:22:21,299 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=1.0, price=107918.59
2025-10-30 08:22:21,300 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: LONG 0.0185 @ $107918.59 (cash flow: -$2000.00, remaining: $8000.00)
2025-10-30 08:22:21,300 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=0.0185
2025-10-30 08:22:21,302 - rl_trading_lab.environment.portfolio - DEBUG - Position held for 1 bars, need 2 bars minimum
2025-10-30 08:22:21,303 - rl_trading_lab.environment.portfolio - DEBUG - Closing position: current=0.0185, signal=-1.0
2025-10-30 08:22:21,304 - rl_trading

Using cpu device
Wrapping the env in a DummyVecEnv.


2025-10-30 08:22:21,496 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 2365/9953
2025-10-30 08:22:21,502 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=1.0, price=111237.71
2025-10-30 08:22:21,502 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: LONG 0.0180 @ $111237.71 (cash flow: -$2000.00, remaining: $8000.00)
2025-10-30 08:22:21,502 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=0.0180
2025-10-30 08:22:21,504 - rl_trading_lab.environment.trading_env - DEBUG - Hold action closing position: size=0.0180
2025-10-30 08:22:21,504 - rl_trading_lab.environment.portfolio - DEBUG - Position closed: P&L=$0.20, Commission=$0.00, Net=$0.20 (cash flow: +$2000.20, -$0.00, balance: $10000.20)
2025-10-30 08:22:21,505 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 8974/9953
2025-10-30 08:22:21,507 - rl_trading_lab.environment.portfolio - DEBUG - Opening

Eval num_timesteps=500, episode_reward=0.00 +/- 0.00
Episode length: 7142.20 +/- 1718.37
------------------------------------
| eval/                 |          |
|    mean_ep_length     | 7.14e+03 |
|    mean_reward        | 0        |
| time/                 |          |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -1.07    |
|    explained_variance | 0.327    |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | 0.0302   |
|    value_loss         | 0.00171  |
------------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3.62     |
|    ep_rew_mean     | 4.55e-06 |
| time/              |          |
|    fps             | 19       |
|    iterations      | 100      |
|    time_elapsed    | 25       |
|    total_timesteps | 500      |
---------------------------------


2025-10-30 08:22:46,502 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=1.0, price=109979.79
2025-10-30 08:22:46,502 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: LONG 0.0182 @ $109979.79 (cash flow: -$2000.00, remaining: $8000.00)
2025-10-30 08:22:46,503 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=0.0182
2025-10-30 08:22:46,505 - rl_trading_lab.environment.portfolio - DEBUG - Position held for 1 bars, need 2 bars minimum
2025-10-30 08:22:46,507 - rl_trading_lab.environment.portfolio - DEBUG - Closing position: current=0.0182, signal=-1.0
2025-10-30 08:22:46,507 - rl_trading_lab.environment.portfolio - DEBUG - Position closed: P&L=$-0.46, Commission=$0.00, Net=$-0.46 (cash flow: +$1999.54, -$0.00, balance: $9999.54)
2025-10-30 08:22:46,508 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 5957/9953
2025-10-30 08:22:46,513 - rl_trading_lab.environment.portfolio - DEBUG - Opening po

...

Eval num_timesteps=9000, episode_reward=0.00 +/- 0.00
Episode length: 32.20 +/- 53.03
-------------------------------------
| eval/                 |           |
|    mean_ep_length     | 32.2      |
|    mean_reward        | 7.28e-05  |
| time/                 |           |
|    total_timesteps    | 9000      |
| train/                |           |
|    entropy_loss       | -0.972    |
|    explained_variance | -1.58e+03 |
|    learning_rate      | 0.0007    |
|    n_updates          | 1799      |
|    policy_loss        | -0.0076   |
|    value_loss         | 4.52e-05  |
-------------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3.83     |
|    ep_rew_mean     | 1.52e-05 |
| time/              |          |
|    fps             | 54       |
|    iterations      | 1800     |
|    time_elapsed    | 164      |
|    total_timesteps | 9000     |
---------------------------------


2025-10-30 08:25:05,990 - rl_trading_lab.environment.portfolio - DEBUG - Position held for 1 bars, need 2 bars minimum
2025-10-30 08:25:05,993 - rl_trading_lab.environment.portfolio - DEBUG - Closing position: current=-0.0185, signal=1.0
2025-10-30 08:25:05,993 - rl_trading_lab.environment.portfolio - DEBUG - Position closed: P&L=$0.77, Commission=$0.00, Net=$0.77 (cash flow: -$1999.23, balance: $10000.77)
2025-10-30 08:25:05,994 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 4574/9953
2025-10-30 08:25:05,996 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=1.0, price=108147.99
2025-10-30 08:25:05,996 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: LONG 0.0185 @ $108147.99 (cash flow: -$2000.00, remaining: $8000.00)
2025-10-30 08:25:05,996 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=0.0185
2025-10-30 08:25:06,000 - rl_trading_lab.environment.portfolio - DEBUG - Position held for 1

Eval num_timesteps=9500, episode_reward=0.00 +/- 0.00
Episode length: 9.80 +/- 11.29
-------------------------------------
| eval/                 |           |
|    mean_ep_length     | 9.8       |
|    mean_reward        | 1.38e-05  |
| time/                 |           |
|    total_timesteps    | 9500      |
| train/                |           |
|    entropy_loss       | -1.06     |
|    explained_variance | -1.66e+03 |
|    learning_rate      | 0.0007    |
|    n_updates          | 1899      |
|    policy_loss        | 0.00171   |
|    value_loss         | 3.96e-06  |
-------------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3.7      |
|    ep_rew_mean     | 2.09e-06 |
| time/              |          |
|    fps             | 57       |
|    iterations      | 1900     |
|    time_elapsed    | 166      |
|    total_timesteps | 9500     |
---------------------------------


2025-10-30 08:25:07,372 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=1.0, price=112242.10
2025-10-30 08:25:07,373 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: LONG 0.0178 @ $112242.10 (cash flow: -$2000.00, remaining: $8000.00)
2025-10-30 08:25:07,373 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=0.0178
2025-10-30 08:25:07,375 - rl_trading_lab.environment.portfolio - DEBUG - Position held for 1 bars, need 2 bars minimum
2025-10-30 08:25:07,378 - rl_trading_lab.environment.portfolio - DEBUG - Closing position: current=0.0178, signal=-1.0
2025-10-30 08:25:07,378 - rl_trading_lab.environment.portfolio - DEBUG - Position closed: P&L=$0.69, Commission=$0.00, Net=$0.69 (cash flow: +$2000.69, -$0.00, balance: $10000.69)
2025-10-30 08:25:07,379 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 758/9953
2025-10-30 08:25:07,382 - rl_trading_lab.environment.portfolio - DEBUG - Opening posi

Eval num_timesteps=10000, episode_reward=-0.00 +/- 0.00
Episode length: 6.80 +/- 5.27
-------------------------------------
| eval/                 |           |
|    mean_ep_length     | 6.8       |
|    mean_reward        | -8.8e-06  |
| time/                 |           |
|    total_timesteps    | 10000     |
| train/                |           |
|    entropy_loss       | -1.03     |
|    explained_variance | -1.68e+03 |
|    learning_rate      | 0.0007    |
|    n_updates          | 1999      |
|    policy_loss        | -0.00456  |
|    value_loss         | 2.08e-05  |
-------------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3.83     |
|    ep_rew_mean     | 2.41e-05 |
| time/              |          |
|    fps             | 59       |
|    iterations      | 2000     |
|    time_elapsed    | 167      |
|    total_timesteps | 10000    |
---------------------------------



In [32]:
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=10)

2025-10-30 08:12:39,801 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 9906/9953
2025-10-30 08:12:39,803 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=-1.0, price=111639.68
2025-10-30 08:12:39,803 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: SHORT 0.0179 @ $111639.68 (cash flow: +$2000.00, -$0.00, remaining: $12000.00)
2025-10-30 08:12:39,803 - rl_trading_lab.environment.portfolio - DEBUG - Position opened: size=-0.0179
2025-10-30 08:12:39,806 - rl_trading_lab.environment.portfolio - DEBUG - Position held for 1 bars, need 2 bars minimum
2025-10-30 08:12:39,835 - rl_trading_lab.environment.trading_env - DEBUG - Episode reset: randomized start at step 4673/9953
2025-10-30 08:12:39,836 - rl_trading_lab.environment.portfolio - DEBUG - Opening position: signal=-1.0, price=108686.34
2025-10-30 08:12:39,836 - rl_trading_lab.environment.portfolio - DEBUG - Trade #1: SHORT 0.0184 @ $108686.34 (cash flow: +$200

In [33]:
mean_reward, std_reward

(np.float64(-0.0021728999999999993), np.float64(0.003209531880196861))